<a href="https://colab.research.google.com/github/xiabui/aiml/blob/main/MedSAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import os

# 1. Mở hộp thoại upload file trực tiếp từ máy local
print("Vui lòng chọn file xray_dataset.zip từ máy tính của bạn:")
uploaded = files.upload()

# Lấy tên file vừa upload (phòng trường hợp bạn đặt tên khác)
zip_filename = list(uploaded.keys())[0]

# 2. Giải nén dữ liệu vào thư mục /content/dataset
!unzip -q {zip_filename} -d /content/dataset


Vui lòng chọn file xray_dataset.zip từ máy tính của bạn:


Saving images.zip to images.zip
Cloning into 'MedSAM'...
remote: Enumerating objects: 967, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 967 (delta 102), reused 96 (delta 96), pack-reused 842 (from 1)
Receiving objects: 100% (967/967), 62.52 MiB | 18.79 MiB/s, done.
Resolving deltas: 100% (477/477), done.
/content/MedSAM
Obtaining file:///content/MedSAM
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 

In [2]:
!git clone https://github.com/bowang-lab/MedSAM.git
%cd MedSAM
!pip install -e .
!pip install opencv-python matplotlib

Cloning into 'MedSAM'...
remote: Enumerating objects: 967, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 967 (delta 102), reused 96 (delta 96), pack-reused 842 (from 1)
Receiving objects: 100% (967/967), 62.52 MiB | 19.87 MiB/s, done.
Resolving deltas: 100% (477/477), done.
/content/MedSAM/MedSAM
Obtaining file:///content/MedSAM/MedSAM
  Preparing metadata (setup.py) ... done
  Attempting uninstall: medsam
    Found existing installation: medsam 0.0.1
    Uninstalling medsam-0.0.1:
      Successfully uninstalled medsam-0.0.1
  Running setup.py develop for medsam


In [8]:
!wget -O sam_vit_b_01ec64.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

--2026-07-29 04:32:12--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.4, 65.9.168.81, 65.9.168.52, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 375042383 (358M) [binary/octet-stream]
Saving to: ‘sam_vit_b_01ec64.pth’

sam_vit_b_01ec64.pt 100%[===================>] 357.67M   375MB/s    in 1.0s    

2026-07-29 04:32:13 (375 MB/s) - ‘sam_vit_b_01ec64.pth’ saved [375042383/375042383]



In [10]:
import os
import cv2
import torch
import numpy as np
from segment_anything import sam_model_registry, SamPredictor

# --- CẤU HÌNH ---
IMG_DIR = '/content/dataset/images'          # Thư mục chứa ảnh vừa giải nén
LABEL_DIR = '/content/dataset_labels' # Thư mục lưu file .txt kết quả
os.makedirs(LABEL_DIR, exist_ok=True)

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# ĐỔI TÊN FILE CHECKPOINT TẠI ĐÂY
sam_model = sam_model_registry['vit_b'](checkpoint='sam_vit_b_01ec64.pth')
sam_model = sam_model.to(device)
predictor = SamPredictor(sam_model)

def get_bone_bounding_box(image_rgb):
    """
    Hàm tạo prompt Bounding Box tự động.
    Dựa vào việc X-quang xương thường sáng màu (giá trị pixel cao) ở giữa ảnh.
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        # Nếu không tìm thấy, lấy 80% diện tích ảnh làm box mặc định
        h, w = gray.shape
        return np.array([w*0.1, h*0.1, w*0.9, h*0.9])

    # Lấy vùng sáng lớn nhất (khả năng cao là xương đùi)
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    return np.array([x, y, x + w, y + h])

def mask_to_yolo_polygon(mask, img_width, img_height):
    """Chuyển đổi Binary Mask sang mảng tọa độ Polygon của YOLO (0.0 -> 1.0)"""
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return None

    largest_contour = max(contours, key=cv2.contourArea)
    polygon_normalized = []

    # Đơn giản hóa contour để giảm dung lượng file (tương tự Douglas-Peucker)
    epsilon = 0.002 * cv2.arcLength(largest_contour, True)
    approx_contour = cv2.approxPolyDP(largest_contour, epsilon, True)

    for point in approx_contour:
        x_norm = point[0][0] / float(img_width)
        y_norm = point[0][1] / float(img_height)
        polygon_normalized.extend([f"{x_norm:.6f}", f"{y_norm:.6f}"])

    return " ".join(polygon_normalized)

# --- VÒNG LẶP XỬ LÝ ẢNH ---
valid_exts = ['.png', '.jpg', '.jpeg']
image_files = [f for f in os.listdir(IMG_DIR) if os.path.splitext(f)[1].lower() in valid_exts]

print(f"Bắt đầu xử lý {len(image_files)} ảnh trên GPU {device}...")

for img_name in image_files:
    img_path = os.path.join(IMG_DIR, img_name)
    image = cv2.imread(img_path)
    if image is None: continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_height, img_width, _ = image_rgb.shape

    # 1. Đưa ảnh vào mô hình
    predictor.set_image(image_rgb)

    # 2. Lấy prompt box
    input_box = get_bone_bounding_box(image_rgb)

    # 3. Chạy inference lấy Mask
    masks, _, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box[None, :],
        multimask_output=False,
    )

    # 4. Chuyển đổi Mask sang YOLO format (Class ID = 0)
    polygon_str = mask_to_yolo_polygon(masks[0], img_width, img_height)

    # 5. Lưu ra file .txt
    if polygon_str:
        txt_name = os.path.splitext(img_name)[0] + '.txt'
        with open(os.path.join(LABEL_DIR, txt_name), 'w') as f:
            f.write(f"0 {polygon_str}\n")

print("Hoàn tất gán nhãn!")

Bắt đầu xử lý 3631 ảnh trên GPU cuda:0...
Hoàn tất gán nhãn!


In [11]:
from google.colab import files

# 1. Nén thư mục chứa các file .txt kết quả
!zip -r /content/yolo_labels.zip /content/dataset_labels

# 2. Trigger trình duyệt tải file về máy tính của bạn
files.download('/content/yolo_labels.zip')

print("Đang tải file yolo_labels.zip về máy local...")

  adding: content/dataset_labels/ (stored 0%)
  adding: content/dataset_labels/image1_1026_png.rf.464a1ce3899c64cfb3b77776216e777f.txt (deflated 54%)
  adding: content/dataset_labels/image1_930_png.rf.a6ff682719301024c6598ba67c25b346.txt (deflated 64%)
  adding: content/dataset_labels/image2_1774_png.rf.2573380deee9a2e59d64f815df4e3a8e.txt (deflated 66%)
  adding: content/dataset_labels/image1_577_png.rf.5eca49e9eb82c7bf598f4d3d7f354f68.txt (deflated 57%)
  adding: content/dataset_labels/image1_219_png.rf.230f2a3a0bc7108bc0d86c082de74f93.txt (deflated 68%)
  adding: content/dataset_labels/image2_787_png.rf.61bcf9ee8b71f935ac1501f6b1bcaf2e.txt (deflated 72%)
  adding: content/dataset_labels/image1_404_png.rf.e5aa8fe0ff4c322fe4832dd277c787d6.txt (deflated 52%)
  adding: content/dataset_labels/image1_368_png.rf.c8cb52ddeab0e4afa7e7eb21935864a2.txt (deflated 56%)
  adding: content/dataset_labels/image1_1118_png.rf.fac34b31fcfe8244a26131ee967bf7b3.txt (deflated 60%)
  adding: content/datase

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Đang tải file yolo_labels.zip về máy local...


In [13]:
import os
import random
import shutil
from pathlib import Path

# Thư mục gốc
IMG_DIR = '/content/dataset/images'
LABEL_DIR = '/content/dataset_labels'
BASE_DIR = '/content/yolo_dataset'

# Tạo cấu trúc thư mục YOLO
for split in ['train', 'val']:
    os.makedirs(f"{BASE_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{BASE_DIR}/{split}/labels", exist_ok=True)

# Lấy danh sách file hợp lệ (có cả ảnh và file txt)
images = [f for f in os.listdir(IMG_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))]
valid_data = [f for f in images if os.path.exists(os.path.join(LABEL_DIR, os.path.splitext(f)[0] + '.txt'))]

# Chia ngẫu nhiên 80-20
random.shuffle(valid_data)
split_idx = int(len(valid_data) * 0.8)
train_files, val_files = valid_data[:split_idx], valid_data[split_idx:]

def move_files(file_list, split_name):
    for f in file_list:
        base_name = os.path.splitext(f)[0]
        # Copy ảnh
        shutil.copy(os.path.join(IMG_DIR, f), os.path.join(BASE_DIR, split_name, 'images', f))
        # Copy label
        shutil.copy(os.path.join(LABEL_DIR, base_name + '.txt'), os.path.join(BASE_DIR, split_name, 'labels', base_name + '.txt'))

print("Đang chia dữ liệu...")
move_files(train_files, 'train')
move_files(val_files, 'val')

# Tạo file dataset.yaml
yaml_content = f"""
path: {BASE_DIR}
train: train/images
val: val/images

names:
  0: femur
"""
with open('/content/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print(f"Xong! Train: {len(train_files)} ảnh | Val: {len(val_files)} ảnh.")

Đang chia dữ liệu...
Xong! Train: 2902 ảnh | Val: 726 ảnh.


In [15]:
import os
import glob

# Các thư mục cần quét
dirs_to_clean = ['/content/yolo_dataset/train', '/content/yolo_dataset/val']
removed_count = 0

for split_dir in dirs_to_clean:
    labels_dir = os.path.join(split_dir, 'labels')
    images_dir = os.path.join(split_dir, 'images')

    if not os.path.exists(labels_dir):
        continue

    for label_file in os.listdir(labels_dir):
        label_path = os.path.join(labels_dir, label_file)

        with open(label_path, 'r') as f:
            lines = f.readlines()

        valid_lines = []
        for line in lines:
            parts = line.strip().split()
            # Một polygon hợp lệ cần: ClassID + ít nhất 3 điểm (6 tọa độ) = tối thiểu 7 phần tử
            if len(parts) >= 7:
                valid_lines.append(line)

        # Nếu có dòng không hợp lệ
        if len(valid_lines) != len(lines):
            if len(valid_lines) == 0:
                # Xóa file txt
                os.remove(label_path)

                # Xóa ảnh tương ứng
                base_name = label_file.replace('.txt', '')
                for img_path in glob.glob(os.path.join(images_dir, base_name + '.*')):
                    os.remove(img_path)

                removed_count += 1
            else:
                # Ghi đè lại các dòng hợp lệ
                with open(label_path, 'w') as f:
                    f.writelines(valid_lines)

print(f"Đã dọn dẹp xong! Phát hiện và xử lý {removed_count} file nhãn bị lỗi định dạng.")

Đã dọn dẹp xong! Phát hiện và xử lý 20 file nhãn bị lỗi định dạng.


In [14]:
# Cài đặt thư viện
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill

In [16]:
!yolo task=segment mode=train data=/content/dataset.yaml model=yolov8n-seg.pt epochs=50 imgsz=640 batch=32 device=0

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=F

In [19]:
import os
import glob
from google.colab import files

# Tìm tất cả các file best.pt trong thư mục runs/segment
weight_files = glob.glob('/content/runs/segment/*/weights/best.pt')

if not weight_files:
    print("❌ Không tìm thấy bất kỳ file best.pt nào!")
    print("Vui lòng đảm bảo rằng bạn đã chạy lệnh train và quá trình train đã hoàn tất 100%.")
else:
    # Lấy file best.pt được tạo ra gần đây nhất
    latest_best_pt = max(weight_files, key=os.path.getctime)
    print(f"✅ Đã tìm thấy file trọng số mới nhất tại: {latest_best_pt}")

    print("⏳ Đang tiến hành chuyển đổi sang ONNX...")
    # Chạy lệnh export YOLO với đường dẫn động
    os.system(f"yolo export model={latest_best_pt} format=onnx opset=12")

    # Xác định đường dẫn file .onnx vừa tạo
    onnx_path = latest_best_pt.replace('.pt', '.onnx')

    if os.path.exists(onnx_path):
        print(f"✅ Chuyển đổi thành công! Đang tải về máy file: {onnx_path}")
        files.download(onnx_path)
    else:
        print("❌ Quá trình chuyển đổi sang ONNX thất bại. Vui lòng kiểm tra lại log.")


❌ Không tìm thấy bất kỳ file best.pt nào!
Vui lòng đảm bảo rằng bạn đã chạy lệnh train và quá trình train đã hoàn tất 100%.


In [21]:
from google.colab import files

# Export sang định dạng ONNX (opset 12 tối ưu cho web)
!yolo export model=/content/MedSAM/MedSAM/runs/segment/train-2/weights/best.pt format=onnx opset=12

# Tải file về máy
files.download('/content/MedSAM/MedSAM/runs/segment/train-2/weights/best.onnx')
print("Đang tải best.onnx về máy! Đây là file sẽ gắn vào web.")

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs

PyTorch: starting from '/content/MedSAM/MedSAM/runs/segment/train-2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 37, 8400), (1, 32, 160, 160)) (6.4 MB)

ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 1.7s, saved as '/content/MedSAM/MedSAM/runs/segment/train-2/weights/best.onnx' (12.7 MB)

Export complete (2.0s)
Results saved to /content/MedSAM/MedSAM/runs/segment/train-2/weights/best.onnx
Predict:         yolo predict task=segment model=/content/MedSAM/MedSAM/runs/segment/train-2/weights/best.onnx imgsz=640 
Validate:        yolo val task=segment model=/content/MedSAM/MedSAM/runs/segment/train-2/weights/best.onnx imgsz=640 data=/content/dataset.yaml  
Visualize:       https://netron.app
💡 Learn more at https://doc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Đang tải best.onnx về máy! Đây là file sẽ gắn vào web.
